# Pipeline de Migração de Dados de RH — Delta Lake

Este notebook implementa uma pipeline completa de ingestão e manipulação de dados do sistema fictício de RH da **TechCorp**, utilizando **PySpark** com **Delta Lake**.

## Fonte de Dados
Arquivos CSV exportados do sistema legado de RH:
- `funcionarios.csv` — cadastro de colaboradores
- `departamentos.csv` — estrutura organizacional
- `cargos.csv` — tabela de cargos e faixas salariais
- `folha_pagamento.csv` — histórico de pagamentos mensais

## Diagrama ER (simplificado)
```
DEPARTAMENTOS ──< FUNCIONARIOS >── CARGOS
                      │
                      └──< FOLHA_PAGAMENTO
```

## 1. Inicialização da SparkSession com Delta Lake
Configura o Spark com as extensões do Delta Lake habilitadas.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, current_timestamp, lit, when
from pyspark.sql.types import *
from delta import *

# Cria a SparkSession com suporte a Delta Lake
builder = (
    SparkSession.builder
    .appName('Pipeline_RH_DeltaLake')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.sql.warehouse.dir', '/tmp/spark-warehouse')
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('WARN')

print(f'✅ SparkSession iniciada — versão: {spark.version}')

## 2. Leitura dos Dados da Fonte (CSV)
Carrega os arquivos CSV do sistema legado de RH como DataFrames Spark.

In [ ]:
import os

# Caminho base dos dados de origem
DATA_PATH = '../dados'
DELTA_PATH = '/tmp/delta/rh'

# Lê os CSVs com inferência de schema
df_funcionarios  = spark.read.csv(f'{DATA_PATH}/funcionarios.csv',  header=True, inferSchema=True)
df_departamentos = spark.read.csv(f'{DATA_PATH}/departamentos.csv', header=True, inferSchema=True)
df_cargos        = spark.read.csv(f'{DATA_PATH}/cargos.csv',        header=True, inferSchema=True)
df_folha         = spark.read.csv(f'{DATA_PATH}/folha_pagamento.csv', header=True, inferSchema=True)

print(f'👥 Funcionários carregados:   {df_funcionarios.count()} registros')
print(f'🏢 Departamentos carregados: {df_departamentos.count()} registros')
print(f'📋 Cargos carregados:         {df_cargos.count()} registros')
print(f'💰 Folha carregada:           {df_folha.count()} registros')

In [ ]:
# Visualiza a estrutura da tabela de funcionários
print('=== Schema: Funcionários ===')
df_funcionarios.printSchema()
df_funcionarios.show(5, truncate=False)

## 3. Transformação dos Dados
Aplica conversões de tipo e adiciona metadados de controle antes de gravar.

In [ ]:
from pyspark.sql.functions import to_date, current_timestamp, lit

# Converte colunas de data e adiciona coluna de controle de ingestão
df_funcionarios_clean = (
    df_funcionarios
    .withColumn('data_nascimento', to_date(col('data_nascimento'), 'yyyy-MM-dd'))
    .withColumn('data_admissao',   to_date(col('data_admissao'),   'yyyy-MM-dd'))
    .withColumn('salario',         col('salario').cast(DoubleType()))
    .withColumn('ingested_at',     current_timestamp())  # metadado de auditoria
    .withColumn('source_system',   lit('CSV_LEGADO'))    # origem dos dados
)

df_folha_clean = (
    df_folha
    .withColumn('data_pagamento', to_date(col('data_pagamento'), 'yyyy-MM-dd'))
    .withColumn('ingested_at',    current_timestamp())
    .withColumn('source_system',  lit('CSV_LEGADO'))
)

df_departamentos_clean = df_departamentos.withColumn('ingested_at', current_timestamp())
df_cargos_clean        = df_cargos.withColumn('ingested_at', current_timestamp())

print('✅ Transformações aplicadas com sucesso!')

## 4. INSERT — Escrita Inicial nas Tabelas Delta
Grava os DataFrames transformados no formato Delta Lake (criação das tabelas).

In [ ]:
# -------------------------------------------------------
# INSERT: Grava os dados no formato Delta (modo overwrite)
# Cria as tabelas Delta no sistema de arquivos local
# -------------------------------------------------------

# Tabela de Departamentos
df_departamentos_clean.write.format('delta').mode('overwrite').save(f'{DELTA_PATH}/departamentos')
print('🏢 Tabela delta.departamentos criada!')

# Tabela de Cargos
df_cargos_clean.write.format('delta').mode('overwrite').save(f'{DELTA_PATH}/cargos')
print('📋 Tabela delta.cargos criada!')

# Tabela de Funcionários (com particionamento por status)
df_funcionarios_clean.write \
    .format('delta') \
    .mode('overwrite') \
    .partitionBy('status') \
    .save(f'{DELTA_PATH}/funcionarios')
print('👥 Tabela delta.funcionarios criada (particionada por status)!')

# Tabela de Folha de Pagamento
df_folha_clean.write \
    .format('delta') \
    .mode('overwrite') \
    .partitionBy('competencia') \
    .save(f'{DELTA_PATH}/folha_pagamento')
print('💰 Tabela delta.folha_pagamento criada (particionada por competência)!')

In [ ]:
# Registra as tabelas no catálogo Spark para uso com SQL
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS delta_funcionarios
    USING DELTA LOCATION '{DELTA_PATH}/funcionarios'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS delta_folha
    USING DELTA LOCATION '{DELTA_PATH}/folha_pagamento'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS delta_departamentos
    USING DELTA LOCATION '{DELTA_PATH}/departamentos'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS delta_cargos
    USING DELTA LOCATION '{DELTA_PATH}/cargos'
""")

print('✅ Tabelas registradas no catálogo Spark!')

In [ ]:
# Verifica os dados inseridos
spark.sql('SELECT funcionario_id, nome, cargo_id, salario, status FROM delta_funcionarios ORDER BY funcionario_id').show()

## 5. INSERT — Adicionando Novos Registros
Simula a chegada de novos funcionários e usa `append` para inserir na tabela Delta existente.

In [ ]:
from pyspark.sql import Row
from datetime import date

# Novos funcionários contratados em 2024
novos_funcionarios = [
    Row(funcionario_id=16, nome='Renata Gomes', email='renata.gomes@techcorp.com',
        cpf='666.777.888-99', data_nascimento=date(1998, 4, 12),
        data_admissao=date(2024, 3, 1), departamento_id=1, cargo_id=1,
        salario=5000.0, status='ATIVO', ingested_at=None, source_system='SISTEMA_RH_2024'),
    Row(funcionario_id=17, nome='Gustavo Prado', email='gustavo.prado@techcorp.com',
        cpf='777.888.999-00', data_nascimento=date(1990, 9, 27),
        data_admissao=date(2024, 3, 15), departamento_id=3, cargo_id=4,
        salario=10500.0, status='ATIVO', ingested_at=None, source_system='SISTEMA_RH_2024'),
]

df_novos = spark.createDataFrame(novos_funcionarios) \
    .withColumn('ingested_at', current_timestamp())

# INSERT: modo append adiciona ao Delta existente sem apagar dados anteriores
df_novos.write.format('delta').mode('append').partitionBy('status').save(f'{DELTA_PATH}/funcionarios')

total = spark.read.format('delta').load(f'{DELTA_PATH}/funcionarios').count()
print(f'✅ Novos funcionários inseridos! Total agora: {total} registros')

## 6. UPDATE — Atualizando Salários com DeltaTable API
Usa a API `DeltaTable` para atualizar registros de forma eficiente, sem reescrever toda a tabela.

In [ ]:
from delta.tables import DeltaTable

# Carrega a tabela Delta pelo caminho
delta_funcionarios = DeltaTable.forPath(spark, f'{DELTA_PATH}/funcionarios')

# -------------------------------------------------------
# UPDATE: Reajuste salarial de 10% para analistas sêniores
# Apenas funcionários ATIVOS com cargo_id = 3 (Sênior) são afetados
# -------------------------------------------------------
delta_funcionarios.update(
    condition = "cargo_id = 3 AND status = 'ATIVO'",
    set = {
        'salario': 'salario * 1.10',          # aumento de 10%
        'ingested_at': 'current_timestamp()'  # atualiza metadado de auditoria
    }
)

print('✅ UPDATE executado — salários dos Analistas Sênior reajustados em 10%!')

# Confere o resultado
spark.read.format('delta').load(f'{DELTA_PATH}/funcionarios') \
    .filter("cargo_id = 3") \
    .select('nome', 'cargo_id', 'salario', 'status') \
    .show()

In [ ]:
# UPDATE via SQL — muda o status de funcionários inativos com mais de 5 anos
spark.sql("""
    UPDATE delta_funcionarios
    SET status = 'DESLIGADO',
        ingested_at = current_timestamp()
    WHERE status = 'INATIVO'
    AND datediff(current_date(), data_admissao) > 1825
""")

print('✅ UPDATE via SQL — funcionários inativos há mais de 5 anos marcados como DESLIGADO')
spark.sql("SELECT nome, status, data_admissao FROM delta_funcionarios WHERE status = 'DESLIGADO'").show()

## 7. DELETE — Removendo Registros com DeltaTable API
Remove registros de forma seletiva, mantendo o histórico de versões do Delta.

In [ ]:
# Antes do DELETE: conta registros desligados
antes = spark.read.format('delta').load(f'{DELTA_PATH}/funcionarios').count()
print(f'Registros antes do DELETE: {antes}')

# -------------------------------------------------------
# DELETE: Remove definitivamente os funcionários DESLIGADOS
# (em cenários reais, pode-se preferir soft delete via status)
# -------------------------------------------------------
delta_funcionarios.delete(condition="status = 'DESLIGADO'")

depois = spark.read.format('delta').load(f'{DELTA_PATH}/funcionarios').count()
print(f'Registros após o DELETE:   {depois}')
print(f'🗑️  {antes - depois} registro(s) removido(s)!')

In [ ]:
# DELETE via SQL — remove folhas de pagamento com competência anterior a 2024
spark.sql("""
    DELETE FROM delta_folha
    WHERE competencia < '2024-01'
""")
print('✅ DELETE via SQL — registros de competência anterior a 2024 removidos')

## 8. MERGE (UPSERT) — Sincronização Incremental
O `MERGE` é um dos recursos mais poderosos do Delta Lake para pipelines incrementais.

In [ ]:
# Simula dados chegando do sistema de RH online (atualização + novos)
dados_incremental = [
    # Funcionário existente com salário atualizado
    Row(funcionario_id=1, nome='Ana Paula Silva', email='ana.silva@techcorp.com',
        cpf='123.456.789-01', data_nascimento=date(1990, 3, 15),
        data_admissao=date(2018, 6, 1), departamento_id=1, cargo_id=3,
        salario=9200.0, status='ATIVO', ingested_at=None, source_system='API_RH'),
    # Funcionário novo
    Row(funcionario_id=18, nome='Sofia Barros', email='sofia.barros@techcorp.com',
        cpf='888.999.000-11', data_nascimento=date(2000, 6, 18),
        data_admissao=date(2024, 4, 1), departamento_id=4, cargo_id=1,
        salario=4800.0, status='ATIVO', ingested_at=None, source_system='API_RH'),
]

df_incremental = spark.createDataFrame(dados_incremental) \
    .withColumn('ingested_at', current_timestamp())

# MERGE: atualiza se o ID existe, insere se não existe
delta_funcionarios.alias('target').merge(
    df_incremental.alias('source'),
    'target.funcionario_id = source.funcionario_id'
).whenMatchedUpdate(set={
    'salario':     'source.salario',
    'email':       'source.email',
    'ingested_at': 'current_timestamp()'
}).whenNotMatchedInsertAll().execute()

print('✅ MERGE (UPSERT) executado!')
total = spark.read.format('delta').load(f'{DELTA_PATH}/funcionarios').count()
print(f'Total após MERGE: {total} registros')

## 9. Time Travel — Viagem no Tempo
Delta Lake mantém histórico de versões, permitindo consultar snapshots anteriores.

In [ ]:
# Exibe o histórico de operações na tabela
print('=== Histórico de versões da tabela delta.funcionarios ===')
delta_funcionarios.history().select(
    'version', 'timestamp', 'operation', 'operationMetrics'
).show(truncate=False)

In [ ]:
# Time Travel: lê a versão 0 (dados originais do CSV)
df_versao_original = spark.read \
    .format('delta') \
    .option('versionAsOf', 0) \
    .load(f'{DELTA_PATH}/funcionarios')

print(f'Versão 0 (dados originais): {df_versao_original.count()} registros')
df_versao_original.select('nome', 'salario', 'status').show(5)

## 10. Consulta Analítica Final
Gera um relatório consolidado com os dados migrados e transformados.

In [ ]:
# Relatório: headcount e média salarial por departamento
spark.sql("""
    SELECT 
        d.nome_departamento,
        d.sigla,
        COUNT(f.funcionario_id) AS headcount,
        ROUND(AVG(f.salario), 2) AS salario_medio,
        ROUND(SUM(f.salario), 2) AS folha_total
    FROM delta_funcionarios f
    JOIN delta_departamentos d ON f.departamento_id = d.departamento_id
    WHERE f.status = 'ATIVO'
    GROUP BY d.nome_departamento, d.sigla
    ORDER BY folha_total DESC
""").show()

print('\n✅ Pipeline Delta Lake concluída com sucesso!')

In [ ]:
# Encerra a SparkSession
spark.stop()
print('🔴 SparkSession encerrada.')